In [1]:
"""
%pip install qiskit-ibm-runtime==0.47.0
%pip install samplomatic==0.18.0
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install pylatexenc==2.10
%pip install qiskit==2.3.0
%pip install numpy==2.2.6
%pip install pyscf==2.12.1
%pip install mitiq==1.0.0
%pip install ply==3.11
%pip install openpyxl
"""

'\n%pip install qiskit-ibm-runtime==0.47.0\n%pip install samplomatic==0.18.0\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install pylatexenc==2.10\n%pip install qiskit==2.3.0\n%pip install numpy==2.2.6\n%pip install pyscf==2.12.1\n%pip install mitiq==1.0.0\n%pip install ply==3.11\n%pip install openpyxl\n'

In [2]:
import os
import ast
import json
import time
import warnings
import traceback
import numpy as np
import pandas as pd
from qiskit_aer import AerSimulator
from importlib.metadata import version
from qiskit_ibm_runtime import EstimatorV2
from qiskit_nature.units import DistanceUnit
from qiskit.primitives import StatevectorEstimator
from joblib import Parallel, delayed, parallel_config
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_ibm_runtime.fake_provider import FakeBoston
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Miqit ZNE functions
from mitiq.zne.scaling import fold_global
from mitiq.zne.inference import (
    LinearFactory,
    PolyFactory,
    ExpFactory,
    RichardsonFactory,
    AdaExpFactory,
)

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


# 1°: Building the Molecular Problem


In [3]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees.

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0, 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [4]:
# The original space has approximately 6 electrons, 7 space orbitals, and 14
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)

reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

# 2°: The Hamiltonian in Terms of Qubits

In [5]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [6]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()

qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

In [7]:
# Finally, we obtain the number of qubits and operator in terms of Pauli matrices.
num_qubits = qubit_op.num_qubits
print(f"Number of qubits = {num_qubits}")
print(f"Hamiltonian: {qubit_op}")

Number of qubits = 6
Hamiltonian: SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIIZZ', 'IIIZII', 'IIIZIZ', 'IIZIII', 'IIZIIZ', 'IZIIII', 'IZIIIZ', 'ZIIIII', 'ZIIIIZ', 'IYYIYY', 'IXXIYY', 'IYYIXX', 'IXXIXX', 'YZYYZY', 'XZXYZY', 'YZYXZX', 'XZXXZX', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'YYIYYI', 'XXIYYI', 'YYIXXI', 'XXIXXI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[-2.86740153+0.j,  0.32161792+0.j,  0.31034865+0.j,  0.06196671+0.j,
  0.12889394+0.j,  0.08001574+0.j,  0.32161792+0.j,  0.09975793+0.j,
  0.31034865+0.j,  0.10309655+0.j,  0.12889394+0.j,  0.09238729+0.j,
  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,
  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,
  0.08557179+0.j,  0.10309655+0.j,  0.10888574+0.j,  0.08924797+0.j,
  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,
  0.09238729+0.j,  0.08924797+0.j,  0.11246476+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.08557179+0.j])


# 3°: Ansatz Circuit Construction


In [8]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [9]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

# 4°: Transpilation and Simulator Settings for Noise and Noise Free

In [10]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa_qpu(backend, initial_layout, seed=14):
  target = backend.target

  pm = generate_preset_pass_manager(
       target=target,
       initial_layout=initial_layout,
       optimization_level=3,
       seed_transpiler=seed,
       layout_method='sabre',
       routing_method='sabre'
  )

  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

In [11]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend):
  target = backend.target
  pm = generate_preset_pass_manager(target=target)
  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

# 5°: Measuring Eigenvalues and Energies

In [12]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 6°: Experimental Hyperparameters Settings

In [13]:
maxiter = 1000
shots = 10_000
runs = 50
seed_sim = 42
initial_layout= [43, 42, 56, 63, 62, 61]

# 7°: Importing Benchmarking Datasets

In [14]:
path = "/content/drive/MyDrive/Química Quântica Computacional/Simulações VQE/Simulations/Data/Beryllium Hydride (BeH2)/Hardware Noise Random Init Simulations"

In [15]:
adam_df = pd.read_csv(path+"/BEH2_VQE_ADAM_HARDWARE_NOISE_RANDOM_INIT.csv")
bobyqa_df = pd.read_csv(path+"/BEH2_VQE_BOBYQA_HARDWARE_NOISE_RANDOM_INIT.csv")
cobyla_df = pd.read_csv(path+"/BEH2_VQE_COBYLA_HARDWARE_NOISE_RANDOM_INIT.csv")
gd_df = pd.read_csv(path+"/BEH2_VQE_GD_HARDWARE_NOISE_RANDOM_INIT.csv")
gpso_df = pd.read_csv(path+"/BEH2_VQE_GPSO_HARDWARE_NOISE_RANDOM_INIT.csv")
imfil_df = pd.read_csv(path+"/BEH2_VQE_IMFIL_HARDWARE_NOISE_RANDOM_INIT.csv")
lbfgsb_df = pd.read_csv(path+"/BEH2_VQE_L_BFGS_B_HARDWARE_NOISE_RANDOM_INIT.csv")
qnspsa_df = pd.read_csv(path+"/BEH2_VQE_QNSPSA_HARDWARE_NOISE_RANDOM_INIT.csv")
qpso_df = pd.read_csv(path+"/BEH2_VQE_QPSO_HARDWARE_NOISE_RANDOM_INIT.csv")
spsa_df = pd.read_csv(path+"/BEH2_VQE_SPSA_HARDWARE_NOISE_RANDOM_INIT.csv")

In [16]:
optimizers_df = {
    "ADAM": adam_df,
    "BOBYQA": bobyqa_df,
    "COBYLA": cobyla_df,
    "GD": gd_df,
    "GPSO": gpso_df,
    "IMFIL": imfil_df,
    "LBFGSB": lbfgsb_df,
    "QNSPSA": qnspsa_df,
    "QPSO": qpso_df,
    "SPSA": spsa_df
}

In [17]:
def json_to_list(dataframe):
    treated_df = dataframe.copy()

    json_columns = [
        'optimal_params_json',
        'energies_trajectory_json',
        'params_trajectory_json'
    ]

    for column in json_columns:
        if column in treated_df.columns:
            new_column = column.replace("_json", "")

            treated_df[new_column] = treated_df[column].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

            treated_df = treated_df.drop(columns=[column])

    return treated_df

In [18]:
optimizers_df = {nome: json_to_list(df) for nome, df in optimizers_df.items()}

In [19]:
optimizers_df = pd.concat(optimizers_df.values(), keys=optimizers_df.keys(), names=['Optimizer', 'Original_Index']).reset_index(level='Optimizer')

# 8°: Cleaning Dataset

In [20]:
remove_coluns = ['optimizer', 'steps_requested',
       'steps_done', 'cost_function_evaluation', 'energies_trajectory_len',
       'execution_time','energies_trajectory', 'params_trajectory', 'gpso_seed']

In [21]:
optimizers_df = optimizers_df.drop(columns=remove_coluns, errors='ignore')

# 9°: Optimal Parameters Revaluation in Noise Free Simulator

In [22]:
def evaluate_ideal_energy(optimizer_name, run_idx, optimal_params):
    # Simple ideal statevector backend
    backend = AerSimulator(method='statevector', device='CPU')

    # Pure transpile_to_isa using the initial layout previously defined
    ansatz_isa, isa_observables = transpile_to_isa(backend)

    # Instantiate the exact, noise-free StatevectorEstimator
    estimator = StatevectorEstimator()

    # Run the estimator with the provided optimal parameters
    job = estimator.run([(ansatz_isa, isa_observables, optimal_params)])
    exp_val = job.result()[0].data.evs

    # Convert the expectation value to total chemical energy (Hartree)
    # This automatically handles "Adding the constant energies" (nuclear repulsion + core energy)
    total_energy = interpret_exp_val(exp_val, reduced_molecule_problem)

    return {
        "Optimizer": optimizer_name,
        "Run": run_idx,
        "Ideal_Energy": float(total_energy)
    }

In [23]:
t0 = time.perf_counter()

In [24]:
# Extract tasks
tasks = []
for index, row in optimizers_df.iterrows():
    # Handle Optimizer name safely whether it's an index or a column
    opt_name = row['Optimizer'] if 'Optimizer' in optimizers_df.columns else index
    run_idx = row['run'] if 'run' in row else 0
    opt_params = row['optimal_params']
    tasks.append((opt_name, run_idx, opt_params))

In [25]:
# Execute in parallel across all available CPU cores
n_jobs = os.cpu_count()
results = Parallel(n_jobs=n_jobs, prefer="threads")(
    delayed(evaluate_ideal_energy)(opt, run, params) for opt, run, params in tasks
)

In [26]:
t1 = time.perf_counter()
print(f"Parallel evaluation completed in {t1 - t0:.2f} seconds.")

# Generate the Raw DataFrame
df_ideal_raw = pd.DataFrame(results)

Parallel evaluation completed in 17.04 seconds.


In [27]:
# Generate the Statistical DataFrame
# Calculate the mean and standard deviation of the ideal energies per optimizer
df_ideal_stats = df_ideal_raw.groupby('Optimizer')['Ideal_Energy'].agg(['mean', 'std']).reset_index()
df_ideal_stats = df_ideal_stats.rename(columns={'mean': 'Mean_Ideal_Energy', 'std': 'Std_Ideal_Energy'})

In [28]:
# Sort by lowest mean energy
df_ideal_stats = df_ideal_stats.sort_values(by='Mean_Ideal_Energy', ascending=True).reset_index(drop=True)

display(df_ideal_stats)

,Optimizer,Mean_Ideal_Energy,Std_Ideal_Energy
0,BOBYQA,-15.561470,0.001405
1,GPSO,-15.561017,0.001560
2,QPSO,-15.560521,0.002201
3,IMFIL,-15.558584,0.019830
4,QNSPSA,-15.558249,0.004488
5,SPSA,-15.550467,0.056838
6,COBYLA,-15.473980,0.091772
7,GD,-15.396989,0.096907
8,LBFGSB,-15.280694,0.147367
9,ADAM,-15.102577,0.108676


# 10°: Zero-Noise Extrapolation Error Mitigation

In [29]:
# ============================================================
# POST-OPTIMIZATION ZERO-NOISE EXTRAPOLATION (ZNE)
# Mitiq 1.0.0 | BeH2 | CAS(4,3) | UCCSD | FakeBoston
#
# Run convention inherited from the original benchmark:
#   run 0  -> seed_sim 42
#   run 1  -> seed_sim 43
#   ...
#   run 49 -> seed_sim 91
#
# Parallelism is performed ACROSS independent runs.
# ============================================================

In [30]:
# ============================================================
# EXPERIMENTAL CONFIGURATION
# ============================================================

ZNE_SHOTS = 40_000

# Fixed grid shared by Linear, Quadratic, Exponential and Richardson ZNE.
FIXED_SCALES = [1.0, 1.5, 2.0, 2.5, 3.0]

# Adaptive exponential configuration.
# These are the Mitiq 1.0.0 defaults, explicitly frozen here.
ADAPTIVE_STEPS = 5
ADAPTIVE_SECOND_SCALE = 2.0
ADAPTIVE_MAX_SCALE = 6.0

EXPECTED_RUNS = 50
FIRST_SIMULATOR_SEED = 42

# Use all CPUs reported by the environment, as in the original benchmark.
N_JOBS = os.cpu_count() or 1

# Independent deterministic namespace for derived ZNE seeds.
ZNE_SEED_NAMESPACE = 20260815

EXPECTED_VERSIONS = {
    "qiskit": "2.3.0",
    "qiskit-aer": "0.17.2",
    "qiskit-ibm-runtime": "0.47.0",
    "qiskit-nature": "0.7.2",
    "qiskit-algorithms": "0.4.0",
    "mitiq": "1.0.0",
}

for package, expected in EXPECTED_VERSIONS.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(f"{package}: expected {expected}, found {actual}.")

"""
if int(np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(
        f"Mitiq 1.0.0 requires NumPy < 2.0, but NumPy {np.__version__} is loaded. "
        "Install numpy==1.26.4 and restart the runtime before the official experiment."
    )
"""

print("=== ZNE CONFIGURATION ===")
print("CPU count:", os.cpu_count())
print("Parallel workers:", N_JOBS)
print("Shots per circuit:", ZNE_SHOTS)
print("Fixed scales:", FIXED_SCALES)
print("Richardson polynomial order:", len(FIXED_SCALES) - 1)
print(
    "Adaptive:",
    {
        "steps": ADAPTIVE_STEPS,
        "second_scale": ADAPTIVE_SECOND_SCALE,
        "max_scale": ADAPTIVE_MAX_SCALE,
    },
)
print("NumPy:", np.__version__)

=== ZNE CONFIGURATION ===
CPU count: 44
Parallel workers: 44
Shots per circuit: 40000
Fixed scales: [1.0, 1.5, 2.0, 2.5, 3.0]
Richardson polynomial order: 4
Adaptive: {'steps': 5, 'second_scale': 2.0, 'max_scale': 6.0}
NumPy: 2.2.6


In [31]:
# ============================================================
# INPUT VALIDATION
# ============================================================

required_columns = {"Optimizer", "run", "seed_sim", "optimal_params"}
missing_columns = required_columns - set(optimizers_df.columns)

if missing_columns:
    raise ValueError(f"optimizers_df is missing required columns: {sorted(missing_columns)}")

if optimizers_df.duplicated(["Optimizer", "run"]).any():
    raise ValueError("Duplicate (Optimizer, run) pairs were detected.")

optimizer_order = optimizers_df["Optimizer"].drop_duplicates().tolist()

expected_runs = list(range(EXPECTED_RUNS))
expected_seeds = [FIRST_SIMULATOR_SEED + run for run in expected_runs]

for optimizer_name in optimizer_order:
    group = optimizers_df.loc[optimizers_df["Optimizer"] == optimizer_name].copy()
    group = group.sort_values("run")

    runs_found = group["run"].astype(int).tolist()
    seeds_found = group["seed_sim"].astype(int).tolist()

    if len(group) != EXPECTED_RUNS:
        raise ValueError(
            f"{optimizer_name}: expected {EXPECTED_RUNS} runs, found {len(group)}."
        )

    if runs_found != expected_runs:
        raise ValueError(
            f"{optimizer_name}: run identifiers must be exactly 0,...,49. "
            f"Found: {runs_found}"
        )

    if seeds_found != expected_seeds:
        raise ValueError(
            f"{optimizer_name}: seed_sim must follow the benchmark mapping "
            f"run 0 -> 42, ..., run 49 -> 91. Found: {seeds_found}"
        )

print("Input validation passed.")
print("Each optimizer contains exactly 50 correctly paired (run, seed_sim, theta*) records.")

Input validation passed.
Each optimizer contains exactly 50 correctly paired (run, seed_sim, theta*) records.


In [32]:
# ============================================================
# BASE HARDWARE-AWARE CIRCUIT
# ============================================================

base_backend = AerSimulator.from_backend(FakeBoston())
base_ansatz_isa, base_isa_observables = transpile_to_isa_qpu(
    base_backend,
    initial_layout
)

print("\n=== BASE CIRCUIT ===")
print("Qubits:", base_ansatz_isa.num_qubits)
print("Parameters:", base_ansatz_isa.num_parameters)
print("Depth:", base_ansatz_isa.depth())
print("Size:", base_ansatz_isa.size())


=== BASE CIRCUIT ===
Qubits: 156
Parameters: 8
Depth: 806
Size: 1073


In [33]:
# ============================================================
# ENERGY CONVERSION
# ============================================================

def active_to_total_energy(active_energy):
    """
    Adds frozen-core and nuclear-repulsion contributions only
    after the ZNE extrapolation.
    """

    if active_energy is None or not np.isfinite(active_energy):
        return np.nan

    return float(
        np.real(
            interpret_exp_val(float(active_energy), reduced_molecule_problem)
        )
    )

In [34]:
# ============================================================
# DETERMINISTIC ZNE SEEDS
# ============================================================

def qem_seed(original_seed, stage, evaluation_index):
    """
    Seed policy:

    1) The native fixed-grid point lambda=1 reuses exactly the
       simulator seed stored in the original benchmark CSV.

    2) Every additional ZNE execution receives a deterministic
       sub-seed derived from the original seed.

    Thus scheduling order does not affect any result.
    """

    original_seed = int(original_seed)

    if stage == "fixed" and evaluation_index == 0:
        return original_seed

    stage_id = {"fixed": 1, "adaptive": 2}[stage]

    seed_sequence = np.random.SeedSequence(
        [ZNE_SEED_NAMESPACE, original_seed, stage_id, int(evaluation_index)]
    )

    return int(seed_sequence.generate_state(1, dtype=np.uint32)[0])

In [35]:
# ============================================================
# CIRCUIT AUDIT
# ============================================================

def two_qubit_gate_count(circuit):
    return int(sum(len(instruction.qubits) == 2 for instruction in circuit.data))

In [36]:
# ============================================================
# PROCESS-LOCAL SIMULATION CONTEXT
# ============================================================

def make_run_context():
    """
    Creates an independent FakeBoston/Aer/Estimator context
    inside each parallel worker.

    Aer itself is restricted to one thread because parallelism
    is performed externally across runs.
    """

    backend = AerSimulator.from_backend(FakeBoston())

    backend.set_options(
        max_parallel_threads=1,
        max_parallel_experiments=1,
        max_parallel_shots=1
    )

    estimator = EstimatorV2(mode=backend)
    estimator.options.default_shots = ZNE_SHOTS

    # Explicitly disable Runtime error mitigation/suppression.
    # Our experiment must measure only the manually implemented Mitiq ZNE.
    estimator.options.resilience_level = 0
    estimator.options.twirling.enable_gates = False
    estimator.options.twirling.enable_measure = False

    # The base circuit is already hardware-aware and physically mapped.
    # After folding, optimization_level=0 is used only to translate
    # inverse/folded gates back into the target ISA.
    #
    # Aggressive optimization must not cancel the intentionally inserted
    # U U^\dagger folding structure.
    identity_layout = list(range(base_ansatz_isa.num_qubits))

    fold_pass_manager = generate_preset_pass_manager(
        target=backend.target,
        optimization_level=0,
        initial_layout=identity_layout,
        seed_transpiler=14
    )

    return backend, estimator, fold_pass_manager

In [37]:
# ============================================================
# PARAMETER BINDING
# ============================================================

def bind_optimal_parameters(task):
    optimizer_name = str(task["Optimizer"])
    run_idx = int(task["run"])
    optimal_params = np.asarray(task["optimal_params"], dtype=float)

    if optimal_params.size != base_ansatz_isa.num_parameters:
        raise ValueError(
            f"{optimizer_name} run {run_idx}: received {optimal_params.size} parameters, "
            f"but the circuit expects {base_ansatz_isa.num_parameters}."
        )

    bound_circuit = base_ansatz_isa.assign_parameters(optimal_params, inplace=False)

    if bound_circuit.num_parameters != 0:
        raise RuntimeError(
            f"{optimizer_name} run {run_idx}: free parameters remain after binding."
        )

    return bound_circuit

In [38]:
# ============================================================
# NOISE SCALING
# ============================================================

def prepare_scaled_circuit(bound_circuit, scale_factor, fold_pass_manager):
    """
    Generates the executable circuit associated with a nominal
    ZNE noise scale lambda.
    """

    # Native circuit. No folding and no second transpilation.
    if np.isclose(scale_factor, 1.0):
        return bound_circuit

    # Global unitary folding is applied to the already compiled circuit.
    folded_circuit = fold_global(
        bound_circuit,
        scale_factor=float(scale_factor)
    )

    # Translate the inverse operations introduced by folding to the
    # FakeBoston target instruction set without optimization.
    return fold_pass_manager.run(folded_circuit)

In [39]:
# ============================================================
# NOISY CIRCUIT EXECUTION
# ============================================================

def evaluate_active_energy(circuit, backend, estimator, seed_simulator):
    """
    Evaluates only the active-space expectation value.

    The seed is explicitly applied to both the Aer backend and
    Runtime Estimator simulator options.
    """

    seed_simulator = int(seed_simulator)

    backend.set_options(seed_simulator=seed_simulator)
    estimator.options.seed_estimator = seed_simulator
    estimator.options.simulator.seed_simulator = seed_simulator

    job = estimator.run([(circuit, base_isa_observables)])
    expectation_value = job.result()[0].data.evs

    return float(np.real(np.asarray(expectation_value)).item())

In [40]:
# ============================================================
# NON-ADAPTIVE FIT
# ============================================================

def fit_nonadaptive(factory, scale_factors, expectation_values):
    """
    Fits one Mitiq non-adaptive extrapolation model.

    Returned uncertainty is the Mitiq fit uncertainty, not the
    full physical/statistical uncertainty of the mitigated result.
    """

    for scale, value in zip(scale_factors, expectation_values):
        factory.push({"scale_factor": float(scale)}, float(value))

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            estimate = float(factory.reduce())

        try:
            fit_uncertainty = factory.get_zero_noise_limit_error()
            fit_uncertainty = np.nan if fit_uncertainty is None else float(fit_uncertainty)
        except Exception:
            fit_uncertainty = np.nan

        warning_message = " | ".join(str(w.message) for w in caught) or None
        return estimate, fit_uncertainty, warning_message, None

    except Exception as exc:
        return np.nan, np.nan, None, repr(exc)

In [41]:
# ============================================================
# STAGE 1 — FIXED-GRID ZNE
#
# One complete run:
#
# lambda = 1.0, 1.5, 2.0, 2.5, 3.0
#
# The SAME five measurements are used for:
#   - Linear
#   - Quadratic
#   - Exponential
#   - Richardson
# ============================================================

def evaluate_fixed_run(task):
    optimizer_name = str(task["Optimizer"])
    run_idx = int(task["run"])
    original_seed = int(task["seed_sim"])

    bound_circuit = bind_optimal_parameters(task)
    backend, estimator, fold_pass_manager = make_run_context()

    active_energies = []
    seeds_used = []
    depths = []
    sizes = []
    two_q_counts = []

    for scale_index, scale_factor in enumerate(FIXED_SCALES):
        executable_circuit = prepare_scaled_circuit(
            bound_circuit,
            scale_factor,
            fold_pass_manager
        )

        seed = qem_seed(original_seed, "fixed", scale_index)

        active_energy = evaluate_active_energy(
            executable_circuit,
            backend,
            estimator,
            seed
        )

        active_energies.append(active_energy)
        seeds_used.append(seed)
        depths.append(int(executable_circuit.depth()))
        sizes.append(int(executable_circuit.size()))
        two_q_counts.append(two_qubit_gate_count(executable_circuit))

    linear = fit_nonadaptive(
        LinearFactory(scale_factors=FIXED_SCALES),
        FIXED_SCALES,
        active_energies
    )

    quadratic = fit_nonadaptive(
        PolyFactory(scale_factors=FIXED_SCALES, order=2),
        FIXED_SCALES,
        active_energies
    )

    exponential = fit_nonadaptive(
        ExpFactory(scale_factors=FIXED_SCALES, asymptote=None),
        FIXED_SCALES,
        active_energies
    )

    richardson = fit_nonadaptive(
        RichardsonFactory(scale_factors=FIXED_SCALES),
        FIXED_SCALES,
        active_energies
    )

    optimization_final_energy = np.nan

    if "final_energy" in task and task["final_energy"] is not None:
        try:
            if not pd.isna(task["final_energy"]):
                optimization_final_energy = float(task["final_energy"])
        except Exception:
            pass

    return {
        "Optimizer": optimizer_name,
        "Run": run_idx,
        "Original_Seed_Sim": original_seed,
        "ZNE_Shots": ZNE_SHOTS,
        "Optimization_Final_Energy": optimization_final_energy,

        # Native lambda=1 reevaluation at 40,000 shots
        "Raw_Active_Energy": active_energies[0],

        # ZNE estimates
        "ZNE_Linear_Active_Energy": linear[0],
        "ZNE_Quadratic_Active_Energy": quadratic[0],
        "ZNE_Exponential_Active_Energy": exponential[0],
        "ZNE_Richardson_Active_Energy": richardson[0],

        # Fit uncertainty
        "Linear_Fit_Uncertainty": linear[1],
        "Quadratic_Fit_Uncertainty": quadratic[1],
        "Exponential_Fit_Uncertainty": exponential[1],
        "Richardson_Fit_Uncertainty": richardson[1],

        # Fit diagnostics
        "Linear_Warning": linear[2],
        "Quadratic_Warning": quadratic[2],
        "Exponential_Warning": exponential[2],
        "Richardson_Warning": richardson[2],
        "Linear_Error": linear[3],
        "Quadratic_Error": quadratic[3],
        "Exponential_Error": exponential[3],
        "Richardson_Error": richardson[3],

        # Full reproducibility / circuit audit
        "Fixed_Scales_JSON": json.dumps(FIXED_SCALES),
        "Fixed_Seeds_JSON": json.dumps(seeds_used),
        "Fixed_Active_Energies_JSON": json.dumps(active_energies),
        "Fixed_Depths_JSON": json.dumps(depths),
        "Fixed_Sizes_JSON": json.dumps(sizes),
        "Fixed_TwoQ_JSON": json.dumps(two_q_counts),
    }

In [42]:
# ============================================================
# STAGE 2 — ADAPTIVE EXPONENTIAL ZNE
#
# Mitiq 1.0.0 configuration:
#   lambda_1 = 1
#   lambda_2 = 2
#   lambda_3 = 4
#   lambda_4 and lambda_5 selected adaptively
#   maximum lambda = 6
#
# lambda=1 is reused from Stage 1 and is NOT re-executed.
# ============================================================

def evaluate_adaptive_run(task, native_active_energy):
    optimizer_name = str(task["Optimizer"])
    run_idx = int(task["run"])
    original_seed = int(task["seed_sim"])

    bound_circuit = bind_optimal_parameters(task)
    backend, estimator, fold_pass_manager = make_run_context()

    factory = AdaExpFactory(
        steps=ADAPTIVE_STEPS,
        scale_factor=ADAPTIVE_SECOND_SCALE,
        asymptote=None,
        max_scale_factor=ADAPTIVE_MAX_SCALE
    )

    # Reuse the lambda=1 result produced in Stage 1.
    factory.push({"scale_factor": 1.0}, float(native_active_energy))

    adaptive_seeds = [original_seed]
    depths = [int(bound_circuit.depth())]
    sizes = [int(bound_circuit.size())]
    two_q_counts = [two_qubit_gate_count(bound_circuit)]

    adaptive_estimate = np.nan
    adaptive_fit_uncertainty = np.nan
    adaptive_warning = None
    adaptive_error = None

    try:
        adaptive_index = 0

        while not factory.is_converged():
            next_params = factory.next()
            scale_factor = float(next_params["scale_factor"])

            executable_circuit = prepare_scaled_circuit(
                bound_circuit,
                scale_factor,
                fold_pass_manager
            )

            seed = qem_seed(original_seed, "adaptive", adaptive_index)

            active_energy = evaluate_active_energy(
                executable_circuit,
                backend,
                estimator,
                seed
            )

            factory.push(next_params, active_energy)

            adaptive_seeds.append(seed)
            depths.append(int(executable_circuit.depth()))
            sizes.append(int(executable_circuit.size()))
            two_q_counts.append(two_qubit_gate_count(executable_circuit))

            adaptive_index += 1

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            adaptive_estimate = float(factory.reduce())

        try:
            fit_uncertainty = factory.get_zero_noise_limit_error()
            adaptive_fit_uncertainty = (
                np.nan if fit_uncertainty is None else float(fit_uncertainty)
            )
        except Exception:
            adaptive_fit_uncertainty = np.nan

        adaptive_warning = " | ".join(str(w.message) for w in caught) or None

    except Exception as exc:
        # Failures are retained in the data and never silently replaced.
        adaptive_error = repr(exc)

    adaptive_scales = [float(x) for x in factory.get_scale_factors()]
    adaptive_energies = [float(x) for x in factory.get_expectation_values()]

    max_scale_hits = int(
        sum(np.isclose(scale, ADAPTIVE_MAX_SCALE) for scale in adaptive_scales)
    )

    return {
        "Optimizer": optimizer_name,
        "Run": run_idx,

        "ZNE_Adaptive_Exponential_Active_Energy": adaptive_estimate,

        "Adaptive_Fit_Uncertainty": adaptive_fit_uncertainty,
        "Adaptive_Warning": adaptive_warning,
        "Adaptive_Error": adaptive_error,
        "Adaptive_Max_Scale_Hits": max_scale_hits,

        # Full reproducibility / circuit audit
        "Adaptive_Scales_JSON": json.dumps(adaptive_scales),
        "Adaptive_Seeds_JSON": json.dumps(adaptive_seeds),
        "Adaptive_Active_Energies_JSON": json.dumps(adaptive_energies),
        "Adaptive_Depths_JSON": json.dumps(depths),
        "Adaptive_Sizes_JSON": json.dumps(sizes),
        "Adaptive_TwoQ_JSON": json.dumps(two_q_counts),
    }

In [43]:
# ============================================================
# PARALLEL EXECUTION
#
# For each optimizer:
#
#   1) execute all 50 fixed-grid runs in parallel
#   2) save fixed-grid checkpoint
#   3) execute all 50 adaptive runs in parallel
#   4) save adaptive checkpoint
#   5) merge both stages
#   6) move to the next optimizer
#
# Only independent runs are parallelized.
# ============================================================

all_optimizer_results = []
global_start = time.perf_counter()

for optimizer_name in optimizer_order:
    optimizer_tasks = (
        optimizers_df.loc[optimizers_df["Optimizer"] == optimizer_name]
        .sort_values("run")
        .to_dict(orient="records")
    )

    print("\n" + "=" * 70)
    print(f"{optimizer_name}: {len(optimizer_tasks)} runs")
    print("=" * 70)

    # ========================================================
    # STAGE 1 — FIXED GRID
    # ========================================================

    fixed_start = time.perf_counter()

    with parallel_config(
        backend="loky",
        n_jobs=N_JOBS,
        inner_max_num_threads=1
    ):
        fixed_results = Parallel(
            batch_size=1,
            verbose=10
        )(
            delayed(evaluate_fixed_run)(task)
            for task in optimizer_tasks
        )

    df_fixed = (
        pd.DataFrame(fixed_results)
        .sort_values("Run")
        .reset_index(drop=True)
    )

    fixed_checkpoint = f"BEH2_ZNE_{optimizer_name}_FIXED.csv"
    df_fixed.to_csv(fixed_checkpoint, index=False)

    print(
        f"Fixed-grid finished in "
        f"{time.perf_counter() - fixed_start:.1f} s"
    )
    print("Saved:", fixed_checkpoint)

    # ========================================================
    # STAGE 2 — ADAPTIVE EXPONENTIAL
    # ========================================================

    raw_lookup = df_fixed.set_index("Run")["Raw_Active_Energy"].to_dict()

    if any(not np.isfinite(raw_lookup[run]) for run in expected_runs):
        raise RuntimeError(
            f"{optimizer_name}: at least one native lambda=1 energy is invalid. "
            "Adaptive ZNE will not be started."
        )

    adaptive_start = time.perf_counter()

    with parallel_config(
        backend="loky",
        n_jobs=N_JOBS,
        inner_max_num_threads=1
    ):
        adaptive_results = Parallel(
            batch_size=1,
            verbose=10
        )(
            delayed(evaluate_adaptive_run)(
                task,
                raw_lookup[int(task["run"])]
            )
            for task in optimizer_tasks
        )

    df_adaptive = (
        pd.DataFrame(adaptive_results)
        .sort_values("Run")
        .reset_index(drop=True)
    )

    adaptive_checkpoint = f"BEH2_ZNE_{optimizer_name}_ADAPTIVE.csv"
    df_adaptive.to_csv(adaptive_checkpoint, index=False)

    print(
        f"Adaptive finished in "
        f"{time.perf_counter() - adaptive_start:.1f} s"
    )
    print("Saved:", adaptive_checkpoint)

    # ========================================================
    # MERGE FIXED + ADAPTIVE
    # ========================================================

    df_optimizer = df_fixed.merge(
        df_adaptive,
        on=["Optimizer", "Run"],
        how="left",
        validate="one_to_one"
    )

    # ========================================================
    # ACTIVE-SPACE -> TOTAL MOLECULAR ENERGY
    #
    # Classical constants are added only AFTER extrapolation.
    # ========================================================

    energy_conversion = {
        "Raw_Active_Energy": "Raw_Energy",
        "ZNE_Linear_Active_Energy": "ZNE_Linear_Energy",
        "ZNE_Quadratic_Active_Energy": "ZNE_Quadratic_Energy",
        "ZNE_Exponential_Active_Energy": "ZNE_Exponential_Energy",
        "ZNE_Richardson_Active_Energy": "ZNE_Richardson_Energy",
        "ZNE_Adaptive_Exponential_Active_Energy": "ZNE_Adaptive_Exponential_Energy",
    }

    for active_column, total_column in energy_conversion.items():
        df_optimizer[total_column] = (
            df_optimizer[active_column].apply(active_to_total_energy)
        )

    # ========================================================
    # COMPLETE OPTIMIZER CHECKPOINT
    # ========================================================

    optimizer_checkpoint = f"BEH2_ZNE_{optimizer_name}_CHECKPOINT.csv"
    df_optimizer.to_csv(optimizer_checkpoint, index=False)

    all_optimizer_results.append(df_optimizer)

    print("Saved:", optimizer_checkpoint)


ADAM: 50 runs


[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.
[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed:   48.1s remaining:  7.2min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed:   50.5s remaining:  3.0min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed:   51.7s remaining:  1.7min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed:   53.7s remaining:  1.1min
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed:   54.5s remaining:   39.5s
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed:   56.0s remaining:   24.0s
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed:   57.1s remaining:   12.5s
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed:  1.3min remaining:    4.9s
[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed:  1.3min finished
[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.


Fixed-grid finished in 77.6 s
Saved: BEH2_ZNE_ADAM_FIXED.csv


[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed:   43.8s remaining:  6.6min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed:   49.0s remaining:  2.9min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed:   50.0s remaining:  1.6min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed:   51.6s remaining:  1.0min
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed:   51.8s remaining:   37.5s
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed:   52.3s remaining:   22.4s
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed:   52.6s remaining:   11.5s
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed:  1.3min remaining:    4.9s
[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed:  1.3min finished
[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.


Adaptive finished in 78.6 s
Saved: BEH2_ZNE_ADAM_ADAPTIVE.csv
Saved: BEH2_ZNE_ADAM_CHECKPOINT.csv

BOBYQA: 50 runs


...

[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed:   44.5s remaining:  6.7min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed:   45.0s remaining:  2.7min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed:   45.2s remaining:  1.5min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed:   45.4s remaining:   53.3s
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed:   45.5s remaining:   32.9s
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed:   45.5s remaining:   19.5s
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed:   45.7s remaining:   10.0s
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed:  1.2min remaining:    4.6s
[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed:  1.2min finished
[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.


Fixed-grid finished in 72.9 s
Saved: BEH2_ZNE_SPSA_FIXED.csv


[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed:   38.8s remaining:  5.8min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed:   39.0s remaining:  2.3min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed:   39.3s remaining:  1.3min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed:   39.5s remaining:   46.3s
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed:   39.6s remaining:   28.7s
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed:   39.8s remaining:   17.1s
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed:   46.2s remaining:   10.1s
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed:  1.1min remaining:    4.0s


Adaptive finished in 71.9 s
Saved: BEH2_ZNE_SPSA_ADAPTIVE.csv
Saved: BEH2_ZNE_SPSA_CHECKPOINT.csv


[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed:  1.2min finished


In [44]:
# ============================================================
# COMPLETE ZNE DATASET
# ============================================================

df_zne_raw = (
    pd.concat(all_optimizer_results, ignore_index=True)
    .sort_values(["Optimizer", "Run"])
    .reset_index(drop=True)
)


In [45]:
# ============================================================
# MERGE WITH IDEAL STATEVECTOR REEVALUATION
#
# All comparisons refer to the SAME theta*:
#
#   E_raw(theta*)
#   E_ZNE(theta*)
#   E_ideal(theta*)
# ============================================================

if "df_ideal_raw" in globals():
    ideal_reference = df_ideal_raw[
        ["Optimizer", "Run", "Ideal_Energy"]
    ].copy()

    ideal_reference["Run"] = ideal_reference["Run"].astype(int)

    if ideal_reference.duplicated(["Optimizer", "Run"]).any():
        raise ValueError(
            "Duplicate (Optimizer, Run) pairs were detected in df_ideal_raw."
        )

    df_zne_raw = df_zne_raw.merge(
        ideal_reference,
        on=["Optimizer", "Run"],
        how="left",
        validate="one_to_one"
    )

    if df_zne_raw["Ideal_Energy"].isna().any():
        raise ValueError(
            "Some ZNE runs could not be paired with their Ideal_Energy."
        )

    error_columns = {
        "Raw_Energy": "Raw_Abs_Error_to_Ideal",
        "ZNE_Linear_Energy": "Linear_Abs_Error_to_Ideal",
        "ZNE_Quadratic_Energy": "Quadratic_Abs_Error_to_Ideal",
        "ZNE_Exponential_Energy": "Exponential_Abs_Error_to_Ideal",
        "ZNE_Richardson_Energy": "Richardson_Abs_Error_to_Ideal",
        "ZNE_Adaptive_Exponential_Energy": "Adaptive_Abs_Error_to_Ideal",
    }

    for energy_column, error_column in error_columns.items():
        df_zne_raw[error_column] = np.abs(
            df_zne_raw[energy_column] - df_zne_raw["Ideal_Energy"]
        )

In [46]:
# ============================================================
# FINAL SAVE
# ============================================================

FINAL_FILE = "BEH2_VQE_ZNE_HARDWARE_NOISE_RANDOM_INIT.csv"
df_zne_raw.to_csv(FINAL_FILE, index=False)

In [47]:
# ============================================================
# ENVIRONMENT / PROTOCOL METADATA
# ============================================================

metadata = {
    "versions": {
        package: version(package)
        for package in EXPECTED_VERSIONS
    },
    "numpy": np.__version__,
    "zne_shots": ZNE_SHOTS,
    "fixed_scales": FIXED_SCALES,
    "richardson_order": len(FIXED_SCALES) - 1,
    "adaptive_steps": ADAPTIVE_STEPS,
    "adaptive_second_scale": ADAPTIVE_SECOND_SCALE,
    "adaptive_max_scale": ADAPTIVE_MAX_SCALE,
    "parallel_workers": N_JOBS,
    "seed_namespace": ZNE_SEED_NAMESPACE,
    "seed_policy": (
        "lambda=1 reuses the original benchmark seed_sim; "
        "all additional ZNE executions use deterministic sub-seeds "
        "derived from seed_sim, stage, and evaluation index."
    ),
}

with open("BEH2_VQE_ZNE_METADATA.json", "w") as file:
    json.dump(metadata, file, indent=4)

In [48]:
# ============================================================
# REFERENCE ENERGY
#
# CASCI(4,3) reference reported in Table 1 of the manuscript.
#
# IMPORTANT:
# If the original CASCI calculation is available with more
# numerical precision, prefer that unrounded value here.
# ============================================================

E_CASCI = -15.5633775664


# ============================================================
# ERROR METRICS
#
# Three conceptually different comparisons are retained:
#
# 1) Ideal vs CASCI
#       -> residual optimization / variational error
#
# 2) Raw or ZNE vs Ideal(theta*)
#       -> hardware/noise-estimation error for the SAME state
#
# 3) Raw or ZNE vs CASCI
#       -> total end-to-end energy error
#
# Signed and absolute errors are both retained.
# ============================================================


# ------------------------------------------------------------
# A. IDEAL STATEVECTOR vs CASCI
# ------------------------------------------------------------

df_zne_raw["Ideal_Error_to_CASCI"] = (
    df_zne_raw["Ideal_Energy"] - E_CASCI
)

df_zne_raw["Ideal_Abs_Error_to_CASCI"] = np.abs(
    df_zne_raw["Ideal_Error_to_CASCI"]
)


# ------------------------------------------------------------
# B. RAW ENERGY vs IDEAL(theta*)
# ------------------------------------------------------------

df_zne_raw["Raw_Error_to_Ideal"] = (
    df_zne_raw["Raw_Energy"]
    - df_zne_raw["Ideal_Energy"]
)

df_zne_raw["Raw_Abs_Error_to_Ideal"] = np.abs(
    df_zne_raw["Raw_Error_to_Ideal"]
)


# ------------------------------------------------------------
# C. ZNE ENERGIES vs IDEAL(theta*)
# ------------------------------------------------------------

zne_methods = {
    "Linear": "ZNE_Linear_Energy",
    "Quadratic": "ZNE_Quadratic_Energy",
    "Exponential": "ZNE_Exponential_Energy",
    "Richardson": "ZNE_Richardson_Energy",
}

for method_name, energy_column in zne_methods.items():

    signed_error_column = f"{method_name}_Error_to_Ideal"
    abs_error_column = f"{method_name}_Abs_Error_to_Ideal"

    df_zne_raw[signed_error_column] = (
        df_zne_raw[energy_column]
        - df_zne_raw["Ideal_Energy"]
    )

    df_zne_raw[abs_error_column] = np.abs(
        df_zne_raw[signed_error_column]
    )


# ------------------------------------------------------------
# D. RAW ENERGY vs CASCI
# ------------------------------------------------------------

df_zne_raw["Raw_Error_to_CASCI"] = (
    df_zne_raw["Raw_Energy"] - E_CASCI
)

df_zne_raw["Raw_Abs_Error_to_CASCI"] = np.abs(
    df_zne_raw["Raw_Error_to_CASCI"]
)


# ------------------------------------------------------------
# E. ZNE ENERGIES vs CASCI
# ------------------------------------------------------------

for method_name, energy_column in zne_methods.items():

    signed_error_column = f"{method_name}_Error_to_CASCI"
    abs_error_column = f"{method_name}_Abs_Error_to_CASCI"

    df_zne_raw[signed_error_column] = (
        df_zne_raw[energy_column] - E_CASCI
    )

    df_zne_raw[abs_error_column] = np.abs(
        df_zne_raw[signed_error_column]
    )


# ============================================================
# MITIGATION IMPROVEMENT RELATIVE TO THE SAME IDEAL STATE
#
# Positive value:
#     ZNE reduced the absolute error.
#
# Zero:
#     no improvement.
#
# Negative value:
#     ZNE made the estimate worse.
#
# This is deliberately an ABSOLUTE improvement in Hartree,
# rather than a percentage ratio, to avoid unstable ratios
# when the raw error is close to zero.
# ============================================================

for method_name in zne_methods:

    df_zne_raw[f"{method_name}_Improvement_to_Ideal"] = (
        df_zne_raw["Raw_Abs_Error_to_Ideal"]
        - df_zne_raw[f"{method_name}_Abs_Error_to_Ideal"]
    )


# ============================================================
# DESCRIPTIVE ENERGY SUMMARY
#
# Ideal_Energy is deliberately placed immediately after
# Raw_Energy, as requested.
#
# Count is included so failed extrapolations cannot silently
# disappear from descriptive statistics.
# ============================================================

energy_columns = [
    "Raw_Energy",
    "Ideal_Energy",
    "ZNE_Linear_Energy",
    "ZNE_Quadratic_Energy",
    "ZNE_Exponential_Energy",
    "ZNE_Richardson_Energy",
]

df_zne_stats = (
    df_zne_raw
    .groupby("Optimizer")[energy_columns]
    .agg(["count", "mean", "std", "median"])
)


# ============================================================
# DESCRIPTIVE ERROR SUMMARY
#
# These statistics distinguish:
#
#   optimization error
#   raw hardware/noise error
#   post-ZNE residual error
#   total error to CASCI
# ============================================================

error_columns = [
    # Residual optimizer / variational error
    "Ideal_Error_to_CASCI",
    "Ideal_Abs_Error_to_CASCI",

    # Raw noisy estimator
    "Raw_Error_to_Ideal",
    "Raw_Abs_Error_to_Ideal",
    "Raw_Error_to_CASCI",
    "Raw_Abs_Error_to_CASCI",

    # Linear
    "Linear_Error_to_Ideal",
    "Linear_Abs_Error_to_Ideal",
    "Linear_Error_to_CASCI",
    "Linear_Abs_Error_to_CASCI",
    "Linear_Improvement_to_Ideal",

    # Quadratic
    "Quadratic_Error_to_Ideal",
    "Quadratic_Abs_Error_to_Ideal",
    "Quadratic_Error_to_CASCI",
    "Quadratic_Abs_Error_to_CASCI",
    "Quadratic_Improvement_to_Ideal",

    # Exponential
    "Exponential_Error_to_Ideal",
    "Exponential_Abs_Error_to_Ideal",
    "Exponential_Error_to_CASCI",
    "Exponential_Abs_Error_to_CASCI",
    "Exponential_Improvement_to_Ideal",

    # Richardson
    "Richardson_Error_to_Ideal",
    "Richardson_Abs_Error_to_Ideal",
    "Richardson_Error_to_CASCI",
    "Richardson_Abs_Error_to_CASCI",
    "Richardson_Improvement_to_Ideal",
]

df_zne_error_stats = (
    df_zne_raw
    .groupby("Optimizer")[error_columns]
    .agg(["count", "mean", "std", "median"])
)


# ============================================================
# SUCCESS RATE OF ZNE
#
# Fraction of valid runs in which a ZNE method produced a
# smaller absolute error to E_ideal(theta*) than the raw
# noisy estimator.
#
# Failed fits are NOT counted as successes.
# ============================================================

success_rows = []

for optimizer_name, group in df_zne_raw.groupby("Optimizer"):

    row = {
        "Optimizer": optimizer_name,
        "Total_Runs": len(group),
    }

    for method_name in zne_methods:

        method_error = group[f"{method_name}_Abs_Error_to_Ideal"]
        raw_error = group["Raw_Abs_Error_to_Ideal"]

        valid_mask = method_error.notna() & raw_error.notna()

        n_valid = int(valid_mask.sum())

        n_improved = int(
            (
                method_error[valid_mask]
                < raw_error[valid_mask]
            ).sum()
        )

        row[f"{method_name}_Valid_Runs"] = n_valid
        row[f"{method_name}_Improved_Runs"] = n_improved

        row[f"{method_name}_Improvement_Rate"] = (
            n_improved / n_valid
            if n_valid > 0
            else np.nan
        )

    success_rows.append(row)


df_zne_success = pd.DataFrame(success_rows)


# ============================================================
# SAVE COMPLETE EXCEL WORKBOOK
#
# Sheets:
#
#   Run_Level_Data
#       Every individual result. This is the most important
#       sheet for reproducibility and subsequent statistics.
#
#   Energy_Summary
#       Raw -> Ideal -> ZNE energy descriptive statistics.
#
#   Error_Summary
#       All signed/absolute errors and mitigation gains.
#
#   ZNE_Success_Rate
#       Number and fraction of runs improved by each method.
# ============================================================

EXCEL_FILE = "BEH2_VQE_ZNE_RI_RESULTS.xlsx"

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    df_zne_raw.to_excel(
        writer,
        sheet_name="Run_Level_Data",
        index=False
    )

    df_zne_stats.to_excel(
        writer,
        sheet_name="Energy_Summary"
    )

    df_zne_error_stats.to_excel(
        writer,
        sheet_name="Error_Summary"
    )

    df_zne_success.to_excel(
        writer,
        sheet_name="ZNE_Success_Rate",
        index=False
    )


# ============================================================
# FINAL DIAGNOSTICS
# ============================================================

print("\n" + "=" * 70)
print("ZNE COMPLETE — RANDOM-INIT INITIALIZATION")
print("=" * 70)

print(
    f"Wall time: "
    f"{time.perf_counter() - global_start:.1f} s"
)

print("Final CSV file:", FINAL_FILE)
print("Final Excel file:", EXCEL_FILE)
print("Metadata file: BEH2_VQE_ZNE_METADATA.json")

print(
    "Linear fit failures:",
    int(df_zne_raw["Linear_Error"].notna().sum())
)

print(
    "Quadratic fit failures:",
    int(df_zne_raw["Quadratic_Error"].notna().sum())
)

print(
    "Exponential fit failures:",
    int(df_zne_raw["Exponential_Error"].notna().sum())
)

print(
    "Richardson fit failures:",
    int(df_zne_raw["Richardson_Error"].notna().sum())
)


print("\n" + "=" * 70)
print("ENERGY SUMMARY")
print("=" * 70)

display(df_zne_stats)


print("\n" + "=" * 70)
print("ERROR SUMMARY")
print("=" * 70)

display(df_zne_error_stats)


print("\n" + "=" * 70)
print("ZNE IMPROVEMENT RATE")
print("=" * 70)

display(df_zne_success)

print("\nSaved:", EXCEL_FILE)


ZNE COMPLETE — RANDOM-INIT INITIALIZATION
Wall time: 1452.7 s
Final CSV file: BEH2_VQE_ZNE_HARDWARE_NOISE_RANDOM_INIT.csv
Final Excel file: BEH2_VQE_ZNE_RI_RESULTS.xlsx
Metadata file: BEH2_VQE_ZNE_METADATA.json
Linear fit failures: 0
Quadratic fit failures: 0
Exponential fit failures: 25
Richardson fit failures: 0

ENERGY SUMMARY


Raw_Energy                                 Ideal_Energy             \
               count       mean       std     median        count       mean   
Optimizer                                                                      
ADAM              50 -15.004475  0.091577 -14.992703           50 -15.102577   
BOBYQA            50 -15.389975  0.035344 -15.409591           50 -15.561470   
COBYLA            50 -15.285236  0.076933 -15.272890           50 -15.473980   
GD                50 -15.253851  0.085987 -15.246500           50 -15.396989   
GPSO              50 -15.370260  0.028333 -15.377934           50 -15.561017   
IMFIL             50 -15.385138  0.038419 -15.401720           50 -15.558584   
LBFGSB            50 -15.148594  0.119596 -15.122658           50 -15.280694   
QNSPSA            50 -15.329037  0.037634 -15.316001           50 -15.558249   
QPSO              50 -15.372492  0.030695 -15.386436           50 -15.560521   
SPSA              50 -15.332124  0.054970 -15.315029           50 -15.550467   

                               ZNE_Linear_Energy             ...  \
                std     median             count       mean  ...   
Optimizer                                                    ...   
ADAM       0.108676 -15.080186                50 -15.074251  ...   
BOBYQA     0.001405 -15.561916                50 -15.514538  ...   
COBYLA     0.091772 -15.488322                50 -15.416297  ...   
GD         0.096907 -15.402748                50 -15.359781  ...   
GPSO       0.001560 -15.560544                50 -15.505779  ...   
IMFIL      0.019830 -15.561549                50 -15.511866  ...   
LBFGSB     0.147367 -15.250834                50 -15.245856  ...   
QNSPSA     0.004488 -15.559638                50 -15.485300  ...   
QPSO       0.002201 -15.560503                50 -15.504253  ...   
SPSA       0.056838 -15.562698                50 -15.480166  ...   

          ZNE_Quadratic_Energy            ZNE_Exponential_Energy             \
                           std     median                  count       mean   
Optimizer                                                                     
ADAM                  0.100877 -15.114264                     49 -15.160743   
BOBYQA                0.034080 -15.581499                     46 -15.576846   
COBYLA                0.084790 -15.478894                     50 -15.474195   
GD                    0.099537 -15.405975                     46 -15.410482   
GPSO                  0.024290 -15.594090                     50 -15.599684   
IMFIL                 0.038920 -15.582040                     47 -15.574677   
LBFGSB                0.149534 -15.269052                     46 -15.317438   
QNSPSA                0.032507 -15.550685                     46 -15.554524   
QPSO                  0.034534 -15.590297                     50 -15.594975   
SPSA                  0.069670 -15.542576                     45 -15.550107   

                               ZNE_Richardson_Energy                       \
                std     median                 count       mean       std   
Optimizer                                                                   
ADAM       0.096199 -15.146049                    50 -15.278941  0.332560   
BOBYQA     0.025170 -15.586705                    50 -15.901461  0.400081   
COBYLA     0.082097 -15.478690                    50 -15.731474  0.547039   
GD         0.099104 -15.415245                    50 -15.639625  0.371264   
GPSO       0.030450 -15.599803                    50 -15.768711  0.468225   
IMFIL      0.037149 -15.585804                    50 -15.947931  0.380915   
LBFGSB     0.149394 -15.278699                    50 -15.559792  0.451377   
QNSPSA     0.030624 -15.554954                    50 -15.680628  0.850717   
QPSO       0.044030 -15.593484                    50 -15.864948  0.530576   
SPSA       0.052758 -15.545384                    50 -15.819370  0.760356   

                      
              median  
Optimize


ERROR SUMMARY


Ideal_Error_to_CASCI                                \
                         count      mean       std    median   
Optimizer                                                      
ADAM                        50  0.460801  0.108676  0.483192   
BOBYQA                      50  0.001907  0.001405  0.001462   
COBYLA                      50  0.089397  0.091772  0.075056   
GD                          50  0.166388  0.096907  0.160629   
GPSO                        50  0.002360  0.001560  0.002833   
IMFIL                       50  0.004794  0.019830  0.001828   
LBFGSB                      50  0.282683  0.147367  0.312543   
QNSPSA                      50  0.005128  0.004488  0.003739   
QPSO                        50  0.002856  0.002201  0.002874   
SPSA                        50  0.012911  0.056838  0.000680   

          Ideal_Abs_Error_to_CASCI                                \
                             count      mean       std    median   
Optimizer                                                          
ADAM                            50  0.460801  0.108676  0.483192   
BOBYQA                          50  0.001907  0.001405  0.001462   
COBYLA                          50  0.089397  0.091772  0.075056   
GD                              50  0.166388  0.096907  0.160629   
GPSO                            50  0.002360  0.001560  0.002833   
IMFIL                           50  0.004794  0.019830  0.001828   
LBFGSB                          50  0.282683  0.147367  0.312543   
QNSPSA                          50  0.005128  0.004488  0.003739   
QPSO                            50  0.002856  0.002201  0.002874   
SPSA                            50  0.012911  0.056838  0.000680   

          Raw_Error_to_Ideal            ... Richardson_Error_to_CASCI  \
                       count      mean  ...                       std   
Optimizer                               ...                             
ADAM                      50  0.098102  ...                  0.332560   
BOBYQA                    50  0.171495  ...                  0.400081   
COBYLA                    50  0.188744  ...                  0.547039   
GD                        50  0.143139  ...                  0.371264   
GPSO                      50  0.190757  ...                  0.468225   
IMFIL                     50  0.173445  ...                  0.380915   
LBFGSB                    50  0.132101  ...                  0.451377   
QNSPSA                    50  0.229212  ...                  0.850717   
QPSO                      50  0.188029  ...                  0.530576   
SPSA                      50  0.218343  ...                  0.760356   

                    Richardson_Abs_Error_to_CASCI                      \
             median                         count      mean       std   
Optimizer                                                               
ADAM       0.281703                            50  0.360118  0.246617   
BOBYQA    -0.334115                            50  0.434760  0.289524   
COBYLA    -0.245548                            50  0.459635  0.335423   
GD        -0.125811                            50  0.311312  0.211840   
GPSO      -0.287225                            50  0.413960  0.295630   
IMFIL     -0.362735                            50  0.449747  0.299325   
LBFGSB     0.040638                            50  0.370598  0.252207   
QNSPSA    -0.446288                            50  0.759415  0.386352   
QPSO      -0.389830                            50  0.497600  0.348787   
SPSA      -0.487820                            50  0.686917  0.404383   

                    Richardson_Improvement_to_Ideal                      \
             median                           count      mean       std   
Optimizer                                                                 
ADAM       0.321912                              50 -0.172167  0.212347   
BOBYQA     0.361512                              50 -0.265090  0.264896   
COBYLA     0.4101


ZNE IMPROVEMENT RATE


,Optimizer,Total_Runs,Linear_Valid_Runs,Linear_Improved_Runs,Linear_Improvement_Rate,Quadratic_Valid_Runs,Quadratic_Improved_Runs,Quadratic_Improvement_Rate,Exponential_Valid_Runs,Exponential_Improved_Runs,Exponential_Improvement_Rate,Richardson_Valid_Runs,Richardson_Improved_Runs,Richardson_Improvement_Rate
0,ADAM,50,50,50,1.0,50,47,0.94,49,38,0.775510,50,13,0.26
1,BOBYQA,50,50,50,1.0,50,50,1.00,46,46,1.000000,50,6,0.12
2,COBYLA,50,50,50,1.0,50,50,1.00,50,50,1.000000,50,11,0.22
3,GD,50,50,50,1.0,50,50,1.00,46,46,1.000000,50,8,0.16
4,GPSO,50,50,50,1.0,50,50,1.00,50,50,1.000000,50,9,0.18
5,IMFIL,50,50,50,1.0,50,50,1.00,47,47,1.000000,50,7,0.14
6,LBFGSB,50,50,50,1.0,50,50,1.00,46,45,0.978261,50,9,0.18
7,QNSPSA,50,50,50,1.0,50,50,1.00,46,46,1.000000,50,4,0.08
8,QPSO,50,50,50,1.0,50,50,1.00,50,50,1.000000,50,10,0.20
9,SPSA,50,50,50,1.0,50,50,1.00,45,45,1.000000,50,4,0.08



Saved: BEH2_VQE_ZNE_RI_RESULTS.xlsx
